## Roboflow parameters

In [ ]:
# Represent the name of the current version of training
TRAINING_NAME = "j2cdynamics_yolo26_model"

# Represent the name of the dataset you want to use on roboflow and the version
ROBOFLOW_WORKSPACE = "dutoits-workspace"
ROBOFLOW_PROJECT = "duplo-dataset"
ROBOFLOW_PROJECT_VERSION = 1

## Setup Drive

In [ ]:
import os
from google.colab import drive

HOME = "/content/drive/MyDrive/" + TRAINING_NAME
drive.mount('/content/drive')

print(HOME)

Mounted at /content/drive
/content/drive/MyDrive/j2cdynamics_yolo26_model


Install YOLO26 via Ultralytics

In [ ]:
%pip install "ultralytics>=8.4.0" supervision roboflow
import ultralytics
ultralytics.checks()

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 21.5/107.7 GB disk)


## Load Roboflow API Key

In [ ]:
from google.colab import userdata

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
if not ROBOFLOW_API_KEY:
    raise ValueError("ROBOFLOW_API_KEY environment variable not set")

## Creating Dataset Folders

In [ ]:
!mkdir {HOME}/datasets
%cd {HOME}/datasets

mkdir: cannot create directory ‘/content/drive/MyDrive/j2cdynamics_yolo26_model/datasets’: File exists
/content/drive/MyDrive/j2cdynamics_yolo26_model/datasets


## Load Dataset From Roboflow

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Here you should change the name of the project
project = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT)

version = project.version(ROBOFLOW_PROJECT_VERSION)
dataset = version.download("yolo26")

loading Roboflow workspace...
loading Roboflow project...


In [ ]:
%cd {HOME}

!yolo task=detect mode=train model=yolo26n.pt data={dataset.location}/data.yaml epochs=300 imgsz=640 plots=True rect=True

/content/drive/MyDrive/j2cdynamics_yolo26_model


In [ ]:
!yolo export model=runs/detect/train/weights/best.pt format=ncnn imgsz=640 rect=True

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
WARNING ⚠️ NCNN export does not support end2end models, disabling end2end branch.
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLO26n summary (fused): 146 layers, 2,494,694 parameters, 0 gradients, 5.2 GFLOPs

PyTorch: starting from 'runs/detect/train/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (14.8 MB)
requirements: Ultralytics requirement ['ncnn'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 1 package in 161ms
Prepared 1 package in 176ms
Installed 1 package in 3ms
 + ncnn==1.0.20260114

requirements: AutoUpdate success ✅ 0.8s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

requirements: Ultralytics requirement ['pnnx'] not found, attempting AutoUpdate...
Using Python 3.12.12 enviro

In [ ]:
!yolo task=detect mode=val model={HOME}/runs/detect/train/weights/best.pt data={dataset.location}/data.yaml

Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 1.3±1.2 ms, read: 0.1±0.1 MB/s, size: 37.0 KB)
val: Scanning /content/drive/MyDrive/j2cdynamics_yolo26_model/datasets/duplo-dataset-1/valid/labels.cache... 941 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 941/941 119.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 59/59 3.6s/it 3:33
                   all        941       3010       0.98      0.641      0.715      0.624
Speed: 4.7ms preprocess, 193.2ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to /content/drive/MyDrive/j2cdynamics_yolo26_model/runs/detect/val
💡 Learn more at https://docs.ultralytics.com/modes/val
